# KPI Computation and Final Data Load Prep

## Objectives
- Compute business-facing KPIs from cleaned Airbnb listing data.
- Build load-ready fact and dimension tables for BI/dashboard use.
- Apply reproducible validation checks before export.
- Save curated outputs to `data/processed/final_load`.

## Data Loading and Standardization

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
processed_candidates = [
    project_root / "data" / "processed" / "Airbnb_Cleaned.csv",
    project_root / "data" / "processes" / "Airbnb_Cleaned.csv",
]

csv_path = next((p for p in processed_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Airbnb_Cleaned.csv not found in data/processed or data/processes")

df = pd.read_csv(csv_path)
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

# Normalize booleans for consistent KPI calculations.
for col in ["instant_bookable", "high_availability"]:
    df[col] = df[col].astype(str).str.lower().eq("true")

df.head()

,id,host_id,host_name,host_verified,host_listings_count,borough,neighbourhood,lat,long,room_type,construction_year,instant_bookable,cancellation_policy,minimum_nights,availability_365,high_availability,price,service_fee,total_price,price_tier,number_of_reviews,last_review,review_year,reviews_per_month,review_rate
0,1001254,80014485718,Madaline,unconfirmed,6,Brooklyn,Kensington,40.64749,-73.97237,Private room,2020,False,strict,10,286,True,966.0,193.0,1159.0,Luxury,9,2021-10-19,2021,0.21,4
1,1002102,52335172823,Jenna,verified,2,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,2007,False,moderate,30,228,True,142.0,28.0,170.0,Economy,45,2022-05-21,2022,0.38,4
2,1002403,78829239556,Elise,unconfirmed,1,Manhattan,Harlem,40.80902,-73.94190,Private room,2005,True,flexible,3,352,True,620.0,124.0,744.0,Luxury,0,NaT,0,0.00,5
3,1002755,85098326012,Garry,unconfirmed,1,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,2005,True,moderate,30,322,True,368.0,74.0,442.0,Premium,270,2019-07-05,2019,4.64,4
4,1003689,92037596077,Lyndon,verified,1,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,2009,False,moderate,10,289,True,204.0,41.0,245.0,Mid-range,9,2018-11-19,2018,0.10,3


## Overall KPI Table

In [2]:
kpi_overall = pd.DataFrame(
    {
        "kpi": [
            "total_listings",
            "avg_total_price",
            "median_total_price",
            "avg_review_rate",
            "avg_reviews_per_month",
            "instant_bookable_share",
            "high_availability_share",
            "avg_minimum_nights",
        ],
        "value": [
            int(len(df)),
            float(round(df["total_price"].mean(), 2)),
            float(round(df["total_price"].median(), 2)),
            float(round(df["review_rate"].mean(), 3)),
            float(round(df["reviews_per_month"].mean(), 3)),
            float(round(df["instant_bookable"].mean(), 4)),
            float(round(df["high_availability"].mean(), 4)),
            float(round(df["minimum_nights"].mean(), 2)),
        ],
    }
)

kpi_overall

,kpi,value
0,total_listings,101803.0000
1,avg_total_price,750.4100
2,median_total_price,750.0000
3,avg_review_rate,3.2780
4,avg_reviews_per_month,1.1630
5,instant_bookable_share,0.4974
6,high_availability_share,0.3611
7,avg_minimum_nights,7.9400


## Segment KPI Table

In [3]:
kpi_segment = (
    df.groupby(["borough", "room_type", "price_tier"], as_index=False)
    .agg(
        listings=("id", "count"),
        avg_total_price=("total_price", "mean"),
        median_total_price=("total_price", "median"),
        avg_review_rate=("review_rate", "mean"),
        avg_reviews_per_month=("reviews_per_month", "mean"),
        instant_bookable_share=("instant_bookable", "mean"),
        high_availability_share=("high_availability", "mean"),
    )
)

kpi_segment = kpi_segment.sort_values(["listings", "avg_total_price"], ascending=[False, False])
kpi_segment.head(20)

,borough,room_type,price_tier,listings,avg_total_price,median_total_price,avg_review_rate,avg_reviews_per_month,instant_bookable_share,high_availability_share
35,Manhattan,Entire home/apt,Luxury,13662,1077.689568,1082.00,3.254355,0.903215,0.498316,0.402357
17,Brooklyn,Entire home/apt,Luxury,10655,1082.667155,1082.00,3.237072,1.245921,0.494228,0.340779
25,Brooklyn,Private room,Luxury,10590,1078.757637,1080.00,3.265345,1.025303,0.496601,0.310482
45,Manhattan,Private room,Luxury,8372,1076.712660,1076.00,3.268872,1.217144,0.495819,0.321548
37,Manhattan,Entire home/apt,Premium,6921,540.357588,540.00,3.304147,0.888460,0.495160,0.396619
19,Brooklyn,Entire home/apt,Premium,5386,541.730234,545.00,3.307835,1.232681,0.502971,0.323245
27,Brooklyn,Private room,Premium,5302,543.654040,545.50,3.283101,1.067482,0.503584,0.324217
47,Manhattan,Private room,Premium,4191,545.337733,547.00,3.287282,1.293675,0.483178,0.334049
63,Queens,Private room,Luxury,4041,1086.073660,1086.00,3.321703,1.588958,0.493937,0.387775
36,Manhattan,Entire home/apt,Mid-range,3402,268.471085,266.31,3.302469,0.923501,0.511758,0.385362


## Host and Neighborhood KPI Tables

In [4]:
host_kpi = (
    df.groupby(["host_id", "host_name", "host_verified"], as_index=False)
    .agg(
        host_listings=("id", "count"),
        avg_host_total_price=("total_price", "mean"),
        avg_host_review_rate=("review_rate", "mean"),
        avg_host_reviews_per_month=("reviews_per_month", "mean"),
    )
    .sort_values(["host_listings", "avg_host_total_price"], ascending=[False, False])
)

neighborhood_kpi = (
    df.groupby(["borough", "neighbourhood"], as_index=False)
    .agg(
        listings=("id", "count"),
        avg_total_price=("total_price", "mean"),
        avg_review_rate=("review_rate", "mean"),
    )
    .sort_values(["listings", "avg_total_price"], ascending=[False, False])
)

host_kpi.head(15), neighborhood_kpi.head(15)

(           host_id host_name host_verified  host_listings  \
 1896    1871089091  Sirikarn   unconfirmed              1   
 2836    2731902634      Mary      verified              1   
 3350    3229595936     Esaie      verified              1   
 6710    6428506977    London      verified              1   
 9943    9456551231         B      verified              1   
 10598  10053072671     Denae      verified              1   
 11062  10531887449    Jeremy      verified              1   
 11318  10777689853   Antonin   unconfirmed              1   
 11685  11151122860     Harry      verified              1   
 13213  12644747852      Dana      verified              1   
 14079  13472137802    Koichi   unconfirmed              1   
 19831  19160347853    Steven   unconfirmed              1   
 20727  20000330853     Jason      verified              1   
 21541  20803726299    Silvia      verified              1   
 27524  26650828572       Amy   unconfirmed              1   
 
      

## Fact Table Construction

In [5]:
fact_listings = df.copy()

# Add derived analytical fields for downstream BI use.
fact_listings["occupancy_proxy"] = (365 - fact_listings["availability_365"]) / 365
fact_listings["engagement_score"] = (
    fact_listings["reviews_per_month"].fillna(0) * fact_listings["review_rate"].fillna(0)
)
fact_listings["revenue_proxy"] = fact_listings["total_price"] * (1 + fact_listings["occupancy_proxy"])

fact_columns = [
    "id",
    "host_id",
    "borough",
    "neighbourhood",
    "room_type",
    "cancellation_policy",
    "price_tier",
    "instant_bookable",
    "high_availability",
    "minimum_nights",
    "availability_365",
    "price",
    "service_fee",
    "total_price",
    "number_of_reviews",
    "reviews_per_month",
    "review_rate",
    "review_year",
    "occupancy_proxy",
    "engagement_score",
    "revenue_proxy",
]

fact_listings = fact_listings[fact_columns].rename(columns={"id": "listing_id"})
fact_listings.head()

,listing_id,host_id,borough,neighbourhood,room_type,cancellation_policy,price_tier,instant_bookable,high_availability,minimum_nights,availability_365,price,service_fee,total_price,number_of_reviews,reviews_per_month,review_rate,review_year,occupancy_proxy,engagement_score,revenue_proxy
0,1001254,80014485718,Brooklyn,Kensington,Private room,strict,Luxury,False,True,10,286,966.0,193.0,1159.0,9,0.21,4,2021,0.216438,0.84,1409.852055
1,1002102,52335172823,Manhattan,Midtown,Entire home/apt,moderate,Economy,False,True,30,228,142.0,28.0,170.0,45,0.38,4,2022,0.375342,1.52,233.808219
2,1002403,78829239556,Manhattan,Harlem,Private room,flexible,Luxury,True,True,3,352,620.0,124.0,744.0,0,0.00,5,0,0.035616,0.00,770.498630
3,1002755,85098326012,Brooklyn,Clinton Hill,Entire home/apt,moderate,Premium,True,True,30,322,368.0,74.0,442.0,270,4.64,4,2019,0.117808,18.56,494.071233
4,1003689,92037596077,Manhattan,East Harlem,Entire home/apt,moderate,Mid-range,False,True,10,289,204.0,41.0,245.0,9,0.10,3,2018,0.208219,0.30,296.013699


## Dimension Tables Construction

In [6]:
dim_host = (
    df[["host_id", "host_name", "host_verified", "host_listings_count"]]
    .drop_duplicates(subset=["host_id"])
    .reset_index(drop=True)
)

dim_location = (
    df[["borough", "neighbourhood", "lat", "long"]]
    .drop_duplicates(subset=["borough", "neighbourhood"])
    .reset_index(drop=True)
)

dim_listing_type = (
    df[["room_type", "cancellation_policy", "price_tier"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_time = (
    df[["review_year"]]
    .drop_duplicates()
    .query("review_year > 0")
    .sort_values("review_year")
    .reset_index(drop=True)
)

[len(dim_host), len(dim_location), len(dim_listing_type), len(dim_time)]

[101802, 234, 58, 16]

## Data Quality Checks and Export

In [7]:
quality_checks = {
    "fact_row_count": int(len(fact_listings)),
    "fact_duplicate_listing_ids": int(fact_listings["listing_id"].duplicated().sum()),
    "fact_null_total_price": int(fact_listings["total_price"].isna().sum()),
    "fact_invalid_review_rate": int(((fact_listings["review_rate"] < 0) | (fact_listings["review_rate"] > 5)).sum()),
    "dim_host_duplicate_host_ids": int(dim_host["host_id"].duplicated().sum()),
}

pd.DataFrame(list(quality_checks.items()), columns=["check", "value"])

,check,value
0,fact_row_count,101803
1,fact_duplicate_listing_ids,0
2,fact_null_total_price,0
3,fact_invalid_review_rate,0
4,dim_host_duplicate_host_ids,0


In [8]:
output_dir = project_root / "data" / "processed" / "final_load"
output_dir.mkdir(parents=True, exist_ok=True)

kpi_overall.to_csv(output_dir / "kpi_overall.csv", index=False)
kpi_segment.to_csv(output_dir / "kpi_segment.csv", index=False)
host_kpi.to_csv(output_dir / "kpi_host.csv", index=False)
neighborhood_kpi.to_csv(output_dir / "kpi_neighborhood.csv", index=False)

fact_listings.to_csv(output_dir / "fact_listings.csv", index=False)
dim_host.to_csv(output_dir / "dim_host.csv", index=False)
dim_location.to_csv(output_dir / "dim_location.csv", index=False)
dim_listing_type.to_csv(output_dir / "dim_listing_type.csv", index=False)
dim_time.to_csv(output_dir / "dim_time.csv", index=False)

print(f"Export complete: {output_dir}")
sorted([p.name for p in output_dir.glob("*.csv")])

Export complete: /Users/hardik/DVA-capstone 2/data/processed/final_load


['dim_host.csv',
 'dim_listing_type.csv',
 'dim_location.csv',
 'dim_time.csv',
 'fact_listings.csv',
 'kpi_host.csv',
 'kpi_neighborhood.csv',
 'kpi_overall.csv',
 'kpi_segment.csv']

## Load-Prep Notes for Tableau and Reporting
- Use `fact_listings.csv` as the primary analytical table.
- Join dimensions on business keys (`host_id`, `borough + neighbourhood`, `room_type + cancellation_policy + price_tier`, `review_year`) as needed.
- Start dashboard design with `kpi_overall.csv` and `kpi_segment.csv` for executive summary views.
- Keep this notebook as the single reproducible source for load exports to maintain ETL traceability.